# K03_00 – Datenrepräsentation und Vorverarbeitung – Studentenversion

Diese Fassung ist für die **aktive Mitarbeit im Kurs** gedacht.

Dieses Notebook ist der **Einstieg in Kapitel 3**.  
Wir schauen uns an, wie tabellarische Daten im Machine Learning dargestellt werden und warum Vorverarbeitung oft notwendig ist.

## Lernziele
Nach diesem Notebook können Sie:
- zwischen **Samples**, **Features** und **Label** unterscheiden
- numerische und kategoriale Merkmale erkennen
- **fehlende Werte** identifizieren
- einfache **One-Hot-Kodierung** anwenden
- den Effekt von **Skalierung** verstehen

## 1. Ein kleines Beispieldataset

Wir verwenden einen kleinen fiktiven Datensatz zu Wohnungsangeboten.  
Jede **Zeile** ist ein Sample, jede **Spalte** ein Merkmal.

Die Zielvariable heißt hier `preis_in_tsd`.


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "wohnflaeche": [45, 52, 68, 80, 95, 110, 130, 70],
    "zimmer": [2, 2, 3, 3, 4, 4, 5, 3],
    "lage": ["einfach", "mittel", "mittel", "gut", "gut", "gut", "sehr_gut", "mittel"],
    "baujahr": [1970, 1985, 1995, 2001, np.nan, 2010, 2018, 1998],
    "preis_in_tsd": [120, 145, 180, 220, 260, 310, 420, 195]
})

df


### Merksatz
- **Sample** = eine Beobachtung, also eine Zeile
- **Features** = beschreibende Merkmale
- **Label** = Zielwert, den wir vorhersagen wollen


In [ ]:
X = df.drop(columns="preis_in_tsd")
y = df["preis_in_tsd"]

print("Feature-Matrix X:")
display(X)

print("Zielvariable y:")
display(y)


## Mini-Übung 1
Beantworten Sie kurz:
1. Wie viele Samples enthält der Datensatz?
2. Wie viele Features hat `X`?
3. Welche Spalte ist das Label?


In [ ]:
print("Anzahl Samples:", X.shape[0])
print("Anzahl Features:", X.shape[1])
print("Label-Spalte:", "preis_in_tsd")


## 2. Numerische und kategoriale Merkmale

Nicht alle Spalten haben denselben Typ:
- `wohnflaeche`, `zimmer`, `baujahr` sind **numerisch**
- `lage` ist **kategorial**

Viele klassische ML-Modelle erwarten am Ende **numerische Eingaben**.


In [ ]:
print(df.dtypes)


## 3. Fehlende Werte

In realen Daten sind Einträge oft unvollständig.  
Hier fehlt ein Wert in der Spalte `baujahr`.


In [ ]:
print(df.isna().sum())


### Mini-Übung 2
Welche Strategien wären hier denkbar?
- Zeile löschen
- Spalte löschen
- Wert schätzen (imputieren)

Im nächsten Schritt verwenden wir eine einfache Schätzung mit dem Median.


In [ ]:
df_imputed = df.copy()
median_baujahr = df_imputed["baujahr"].median()
df_imputed["baujahr"] = df_imputed["baujahr"].fillna(median_baujahr)

print("Verwendeter Median:", median_baujahr)
df_imputed


## 4. Kategoriale Merkmale kodieren

Die Spalte `lage` ist Text.  
Damit ein Modell später damit arbeiten kann, kodieren wir sie mit **One-Hot-Encoding**.


In [ ]:
df_encoded = pd.get_dummies(df_imputed, columns=["lage"], dtype=int)
df_encoded


### Beobachtung
Aus **einer** kategorialen Spalte entstehen **mehrere** numerische Spalten.


## 5. Warum ist Skalierung wichtig?

Betrachten wir zwei Merkmale:
- `wohnflaeche` liegt ungefähr zwischen 45 und 130
- `zimmer` liegt nur zwischen 2 und 5

Ohne Skalierung hat `wohnflaeche` numerisch eine viel größere Größenordnung.
Das kann für distanzbasierte Verfahren problematisch sein.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

features_numeric = df_imputed[["wohnflaeche", "zimmer", "baujahr"]]

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

X_standard = pd.DataFrame(
    standard_scaler.fit_transform(features_numeric),
    columns=features_numeric.columns
)

X_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(features_numeric),
    columns=features_numeric.columns
)

print("Originaldaten:")
display(features_numeric.head())

print("Standardisiert:")
display(X_standard.head())

print("Min-Max-skaliert:")
display(X_minmax.head())


## Mini-Übung 3
Prüfen Sie:
1. Welchen Mittelwert hat jede Spalte nach der Standardisierung ungefähr?
2. In welchem Bereich liegen die Werte nach der Min-Max-Skalierung?


In [ ]:
print("Mittelwerte nach Standardisierung:")
print(X_standard.mean().round(6))

print("\nMinima nach Min-Max-Skalierung:")
print(X_minmax.min().round(6))

print("\nMaxima nach Min-Max-Skalierung:")
print(X_minmax.max().round(6))


## 6. Kleine Zusammenfassung

In diesem Notebook haben wir gesehen:
- tabellarische Daten werden in **X** und **y** getrennt
- Merkmale können unterschiedliche Typen haben
- fehlende Werte müssen behandelt werden
- kategoriale Merkmale müssen oft kodiert werden
- Skalierung ist für viele Verfahren methodisch wichtig

## Transferfrage
Warum wäre es problematisch, Textspalten oder fehlende Werte einfach ungeprüft an ein Modell weiterzugeben?
